# FDS Pokémon Battle Prediction

This notebook reproduces our five-view model from the repository. It combines
reconstructed state, species forecasts, tactical coverage, learned attack pressure
and raw observations. The three averaged out-of-fold probabilities give 0.7103
accuracy. These training battles were used during model selection, so this is a
development CV result.

Attach the competition data and enable Internet access. Use a fresh notebook
session. While the repository is private, add a Kaggle secret named `GITHUB_TOKEN`
and grant this notebook access to it.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

repo_url = "https://github.com/facelessM/FDS-Pokemon-Challenge-2026-2027.git"
repo = Path("/kaggle/working/FDS-Pokemon-Challenge-2026-2027")
env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    env["FDS_GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # A public repository can be cloned without a secret.

if not repo.exists():
    with tempfile.TemporaryDirectory() as folder:
        askpass = Path(folder) / "askpass.py"
        askpass.write_text(
            f"#!{sys.executable}\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] '
            'else os.environ.get("FDS_GITHUB_TOKEN", ""))\n'
        )
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
        result = subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo)],
                                env=env, capture_output=True)
        if result.returncode:
            raise RuntimeError("Clone failed. Check Internet access and GITHUB_TOKEN.")
env.pop("FDS_GITHUB_TOKEN", None)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())

In [ ]:
# Install before importing any of the model libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")], check=True)

## Selected submission

Each of the five views gets 20% weight. The first four average LightGBM and
XGBoost; raw observations use LightGBM. We fit fifteen ensembles on the original
8,000-row training folds. Each test battle receives one ensemble prediction per
partition, using the fixed timeline-group routing rule. Their three probabilities
are averaged before thresholding at 0.5.

The next cell trains from the competition files and checks the resulting CSV
against the reference submission. This full run also writes OOF metrics and
per-fold probabilities. Use one worker if notebook memory is limited.

In [ ]:
import hashlib
import json
import shutil

matches = sorted(Path("/kaggle/input").rglob("train.jsonl"))
data_candidates = [p.parent for p in matches if (p.parent / "test.jsonl").exists()
                   and list(p.parent.glob("*type_chart*.csv"))]
if len(data_candidates) != 1:
    raise RuntimeError("Expected one attached competition dataset; set data explicitly if needed.")
data = data_candidates[0]
output = Path("/kaggle/working/fds-output")
subprocess.run([sys.executable, str(repo / "run.py"), "predict", "--data", str(data),
                "--output", str(output), "--jobs", "2"], check=True, cwd=repo)

reference = json.loads((repo / "results/reference.json").read_text())
submission = output / "submission_matched_core_summary.csv"
actual_hash = hashlib.sha256(submission.read_bytes()).hexdigest()
assert actual_hash == reference["submission_sha256"], "Predictions differ from the reference; inspect metrics.json."
destination = Path("/kaggle/working") / submission.name
shutil.copyfile(submission, destination)
metrics = json.loads((output / "metrics.json").read_text())
print("OOF accuracy:", metrics["averaged_oof_accuracy"])
print("Saved:", destination)
print("CSV matches the reference.")